In [127]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración para ver todas las columnas
pd.set_option('display.max_columns', None)

# Cargar datos
df = pd.read_csv('neo_plants.csv')

print("✅ Datos cargados correctamente")
df.info()

✅ Datos cargados correctamente
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Entry_ID                      2323 non-null   int64  
 1   Site_Name                     2292 non-null   object 
 2   Domesticated                  2323 non-null   object 
 3   IRMS_d13C_Collagen            2243 non-null   float64
 4   d15N_Collagen                 1392 non-null   float64
 5   Latitude_N                    2318 non-null   float64
 6   Longitude_E                   2318 non-null   float64
 7   General_Category_or_Family    2323 non-null   object 
 8   C4_Plant                      2323 non-null   object 
 9   Sampled_Element               2200 non-null   object 
 10  Cultural_Horizon              1681 non-null   object 
 11  Species                       2291 non-null   object 
 12  Site_Description              2

In [128]:
# ============================================
# CREAR COLUMNA UNIFICADA DE SITIO ARQUEOLÓGICO
# ============================================
if 'Site_Name' in df.columns and 'Site_Name_Trigo' in df.columns:
    df['Archaeological_Site'] = df['Site_Name'].fillna(df['Site_Name_Trigo'])
    print(f"✅ 'Archaeological_Site' creada. Cobertura: {df['Archaeological_Site'].notna().sum()} / {len(df)} ({df['Archaeological_Site'].notna().sum()/len(df)*100:.1f}%)")
else:
    print("⚠️ Columnas 'Site_Name' y/o 'Site_Name_Trigo' no encontradas")
    print(f"Columnas disponibles: {df.columns.tolist()}")

✅ 'Archaeological_Site' creada. Cobertura: 2292 / 2323 (98.7%)


In [129]:
# Crear trt con las columnas útiles
columnas_utiles = [
    'Entry_ID',
    'Archaeological_Site',        # ← NUEVA LÍNEA
    'IRMS_d13C_Collagen',
    'd15N_Collagen',
    'Latitude_N',
    'Longitude_E',
    'General_Category_or_Family',
    'Sampled_Element',
    'Cultural_Horizon',
    'Species',
    'Genus',
    'Absolute_Chronology',
    'Modern_Country'
]

trt = df[columnas_utiles].copy()

print(f"✅ trt creado: {trt.shape}")
print(f"Columnas: {trt.columns.tolist()}")

# Verificar nulos
print("\n--- Nulos en trt ---")
for col in trt.columns:
    nulos = trt[col].isnull().sum()
    print(f"{col:30} {nulos:4} nulos ({nulos/len(trt)*100:4.1f}%)")

✅ trt creado: (2323, 13)
Columnas: ['Entry_ID', 'Archaeological_Site', 'IRMS_d13C_Collagen', 'd15N_Collagen', 'Latitude_N', 'Longitude_E', 'General_Category_or_Family', 'Sampled_Element', 'Cultural_Horizon', 'Species', 'Genus', 'Absolute_Chronology', 'Modern_Country']

--- Nulos en trt ---
Entry_ID                          0 nulos ( 0.0%)
Archaeological_Site              31 nulos ( 1.3%)
IRMS_d13C_Collagen               80 nulos ( 3.4%)
d15N_Collagen                   931 nulos (40.1%)
Latitude_N                        5 nulos ( 0.2%)
Longitude_E                       5 nulos ( 0.2%)
General_Category_or_Family        0 nulos ( 0.0%)
Sampled_Element                 123 nulos ( 5.3%)
Cultural_Horizon                642 nulos (27.6%)
Species                          32 nulos ( 1.4%)
Genus                             5 nulos ( 0.2%)
Absolute_Chronology             191 nulos ( 8.2%)
Modern_Country                    0 nulos ( 0.0%)


In [130]:
# Filtrar filas que tengan los dos isótopos
trt_clean = trt.dropna(subset=['IRMS_d13C_Collagen', 'd15N_Collagen']).copy()

print(f"\n✅ trt_clean creado: {trt_clean.shape}")
print(f"Registros perdidos: {len(trt) - len(trt_clean)} ({((len(trt)-len(trt_clean))/len(trt)*100):.1f}%)")


✅ trt_clean creado: (1346, 13)
Registros perdidos: 977 (42.1%)


In [131]:
import re

def extraer_rango_fechas(texto):
    """
    Extrae el primer rango de fechas BP de un texto.
    Retorna (fecha_min, fecha_max) en años BP.
    """
    if pd.isna(texto):
        return None, None
    
    texto = str(texto)
    
    # Patrón para rangos tipo "3500 - 3000 BCE" o "3500-3000 BCE"
    patron_rango = r'(\d{3,5})\s*[-–]\s*(\d{3,5})\s*(?:BCE|BC|BP)'
    match = re.search(patron_rango, texto)
    if match:
        min_bp = int(match.group(2))  # el más reciente (menor número)
        max_bp = int(match.group(1))  # el más antiguo (mayor número)
        return max_bp, min_bp
    
    # Patrón para fechas únicas tipo "2500 BCE"
    patron_unica = r'(\d{3,5})\s*(?:BCE|BC|BP)'
    match = re.search(patron_unica, texto)
    if match:
        fecha = int(match.group(1))
        # Asumimos un rango de ±100 años
        return fecha + 100, fecha - 100
    
    return None, None

def asignar_periodo_estandar(fecha_min, fecha_max):
    """
    Asigna periodo arqueológico estándar según años BP.
    
    Rangos revisados para el Mediterráneo:
    - Neolithic:     10000 - 6200 BP  (8000-4200 BCE)
    - Chalcolithic:   6200 - 4500 BP  (4200-2500 BCE)
    - Bronze Age:     4500 - 3100 BP  (2500-1100 BCE)
    - Iron Age:       3100 - 2400 BP  (1100-400 BCE)
    - Historical:     < 2400 BP (post-400 BCE)
    """
    if fecha_min is None or fecha_max is None:
        return 'unknown'
    
    fecha_media = (fecha_min + fecha_max) / 2
    
    if fecha_media >= 6200:
        return 'Neolithic'
    elif fecha_media >= 4500:
        return 'Chalcolithic'
    elif fecha_media >= 3100:
        return 'Bronze_Age'
    elif fecha_media >= 2400:
        return 'Iron_Age'
    else:
        return 'Historical'

# Extraer fechas de Absolute_Chronology
trt_clean['Fecha_Min_BP'], trt_clean['Fecha_Max_BP'] = zip(*trt_clean['Absolute_Chronology'].apply(extraer_rango_fechas))

# Asignar periodos estandarizados
trt_clean['Chronological_Period'] = trt_clean.apply(
    lambda row: asignar_periodo_estandar(row['Fecha_Min_BP'], row['Fecha_Max_BP']), 
    axis=1
)

print("\n--- Periodos estandarizados (Chronological_Period) ---")
print(trt_clean['Chronological_Period'].value_counts())

# Comparar con Cultural_Horizon original (opcional)
print("\n--- Comparación: Chronological_Period vs Cultural_Horizon (simplificado) ---")
def simplificar_cultural(texto):
    if pd.isna(texto):
        return 'unknown'
    texto = str(texto)
    if 'Neolithic' in texto or 'Neolítico' in texto:
        return 'Neolithic'
    if 'Chalcolithic' in texto or 'Calcolítico' in texto:
        return 'Chalcolithic'
    if 'Bronze' in texto or 'Bronce' in texto:
        return 'Bronze_Age'
    if 'Iron' in texto or 'Hierro' in texto:
        return 'Iron_Age'
    return 'Other'

trt_clean['Cultural_Simplified'] = trt_clean['Cultural_Horizon'].apply(simplificar_cultural)

comparacion = pd.crosstab(trt_clean['Chronological_Period'], trt_clean['Cultural_Simplified'], margins=True)
print(comparacion)


--- Periodos estandarizados (Chronological_Period) ---
Chronological_Period
Historical      983
Chalcolithic    193
Bronze_Age      146
Neolithic        24
Name: count, dtype: int64

--- Comparación: Chronological_Period vs Cultural_Horizon (simplificado) ---
Cultural_Simplified   Bronze_Age  Chalcolithic  Neolithic  Other  unknown  \
Chronological_Period                                                        
Bronze_Age                     0           112         12     13        9   
Chalcolithic                   0             0         56     19      118   
Historical                   258             0          0    570      155   
Neolithic                      0             0         12     12        0   
All                          258           112         80    614      282   

Cultural_Simplified    All  
Chronological_Period        
Bronze_Age             146  
Chalcolithic           193  
Historical             983  
Neolithic               24  
All                   134

In [132]:
def cuenca_mediterranea(lat, lon):
    """
    Asigna cuenca mediterránea según coordenadas.
    Límites:
    - Occidental: lon ≤ 11°E (desde Iberia hasta Sicilia/Cerdeña)
    - Central: 11°E < lon ≤ 23°E (Italia hasta Grecia occidental)
    - Oriental: lon > 23°E (Egeo hasta Levante)
    """
    if pd.isna(lat) or pd.isna(lon):
        return 'unknown'
    
    # Fuera del Mediterráneo (rangos aproximados)
    if not (25 <= lat <= 46 and -10 <= lon <= 42):
        return 'other'
    
    if lon <= 11:
        return 'Western_Med'
    elif lon <= 23:
        return 'Central_Med'
    else:
        return 'Eastern_Med'

trt_clean['Mediterranean_Basin'] = trt_clean.apply(
    lambda row: cuenca_mediterranea(row['Latitude_N'], row['Longitude_E']), 
    axis=1
)

print("\n--- Cuencas Mediterráneas ---")
print(trt_clean['Mediterranean_Basin'].value_counts())


--- Cuencas Mediterráneas ---
Mediterranean_Basin
Central_Med    665
Eastern_Med    461
Western_Med    215
unknown          5
Name: count, dtype: int64


In [133]:
# Asegurar nombres en inglés
# Si alguna columna está en español, la renombramos
trt_clean = trt_clean.rename(columns={
    'Periodo': 'Period_Original'  # si existe, por si acaso
})

trt_clean.to_csv('trt_clean.csv', index=False)
print("\n✅ Dataset final guardado en 'data/processed/trt_clean.csv'")
print(f"Dimensiones: {trt_clean.shape}")


✅ Dataset final guardado en 'data/processed/trt_clean.csv'
Dimensiones: (1346, 18)


In [134]:
trt_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1346 entries, 0 to 2322
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Entry_ID                    1346 non-null   int64  
 1   Archaeological_Site         1334 non-null   object 
 2   IRMS_d13C_Collagen          1346 non-null   float64
 3   d15N_Collagen               1346 non-null   float64
 4   Latitude_N                  1341 non-null   float64
 5   Longitude_E                 1341 non-null   float64
 6   General_Category_or_Family  1346 non-null   object 
 7   Sampled_Element             1267 non-null   object 
 8   Cultural_Horizon            1064 non-null   object 
 9   Species                     1341 non-null   object 
 10  Genus                       1346 non-null   object 
 11  Absolute_Chronology         1254 non-null   object 
 12  Modern_Country              1346 non-null   object 
 13  Fecha_Min_BP                1254 non-n

In [135]:
# Filtrar las muestras que el modelo asignó como "Historical"
historical_samples = trt_clean[trt_clean['Chronological_Period'] == 'Historical']

print("="*60)
print("MUESTRAS CLASIFICADAS COMO 'HISTORICAL'")
print("="*60)
print(f"Total: {len(historical_samples)} muestras")
print(f"De ellas, con Cultural_Horizon no nulo: {historical_samples['Cultural_Horizon'].notna().sum()}")

print("\n--- Valores únicos en Cultural_Horizon (Historical) ---")
print(historical_samples['Cultural_Horizon'].value_counts().head(30))

MUESTRAS CLASIFICADAS COMO 'HISTORICAL'
Total: 983 muestras
De ellas, con Cultural_Horizon no nulo: 828

--- Valores únicos en Cultural_Horizon (Historical) ---
Cultural_Horizon
Latial                                  362
Late Bronze Age                         119
El Algar                                103
Middle Bronze Age                        55
Early Bronze Age                         45
Early Bronze Age ; Middle Bronze Age     39
Celtiberian                              26
Final Palace                             20
Late Minoan II                           20
Greek                                    20
Early Jazira                             12
Middle Cypriot                            4
Recent Chasséen                           2
Late Copper Age                           1
Name: count, dtype: int64


In [136]:
import re

def extraer_rango_fechas_correcto(texto):
    """
    Extrae rango de fechas BCE/BC y convierte a BP correctamente.
    Retorna (fecha_min_BP, fecha_max_BP) donde min = más antigua, max = más reciente.
    """
    if pd.isna(texto):
        return None, None
    
    texto = str(texto)
    
    # Buscar rangos tipo "3500 - 3000 BCE" o "2000-1500 BCE"
    # El orden puede ser "antiguo - reciente" o "reciente - antiguo"
    patron = r'(\d{3,5})\s*[-–]\s*(\d{3,5})\s*(?:BCE|BC)'
    match = re.search(patron, texto)
    if match:
        año1 = int(match.group(1))
        año2 = int(match.group(2))
        
        # Convertir a BP
        bp1 = año1 + 1950  # año1 BCE → BP
        bp2 = año2 + 1950  # año2 BCE → BP
        
        # El valor más alto BP es la fecha más antigua
        fecha_min_BP = max(bp1, bp2)
        fecha_max_BP = min(bp1, bp2)
        
        return fecha_min_BP, fecha_max_BP
    
    # Fechas únicas tipo "1500 BCE"
    patron_unica = r'(\d{3,5})\s*(?:BCE|BC)'
    match = re.search(patron_unica, texto)
    if match:
        año = int(match.group(1))
        bp = año + 1950
        # Asumir rango de ±100 años
        return bp + 100, bp - 100
    
    return None, None

def asignar_periodo_corregido(fecha_min_BP, fecha_max_BP):
    """
    Asigna periodo arqueológico basado en años BP corregidos.
    """
    if fecha_min_BP is None or fecha_max_BP is None:
        return 'unknown'
    
    fecha_media = (fecha_min_BP + fecha_max_BP) / 2
    
    # Rangos BP corregidos
    if fecha_media >= 8000:   # ≥6000 BCE
        return 'Neolithic'
    elif fecha_media >= 6000:  # 6000-8000 BP = 4000-6000 BCE
        return 'Neolithic_Late'
    elif fecha_media >= 5200:  # 5200-6000 BP = 3200-4000 BCE
        return 'Chalcolithic'
    elif fecha_media >= 4200:  # 4200-5200 BP = 2200-3200 BCE
        return 'Early_Bronze'
    elif fecha_media >= 3500:  # 3500-4200 BP = 1500-2200 BCE
        return 'Middle_Late_Bronze'
    elif fecha_media >= 3000:  # 3000-3500 BP = 1000-1500 BCE
        return 'Iron_Age_Early'
    elif fecha_media >= 2400:  # 2400-3000 BP = 500-1000 BCE
        return 'Iron_Age_Late'
    elif fecha_media >= 1700:  # 1700-2400 BP = 500 BCE - 250 CE
        return 'Roman'
    else:
        return 'Historical_Other'

# Aplicar corrección
trt_clean['Fecha_Min_BP_corr'], trt_clean['Fecha_Max_BP_corr'] = zip(*trt_clean['Absolute_Chronology'].apply(extraer_rango_fechas_correcto))
trt_clean['Chronological_Period_corr'] = trt_clean.apply(
    lambda row: asignar_periodo_corregido(row['Fecha_Min_BP_corr'], row['Fecha_Max_BP_corr']),
    axis=1
)

print("\n--- Periodos CORREGIDOS ---")
print(trt_clean['Chronological_Period_corr'].value_counts())


--- Periodos CORREGIDOS ---
Chronological_Period_corr
Iron_Age_Late         408
Middle_Late_Bronze    357
Neolithic_Late        212
Chalcolithic          127
Historical_Other       92
Early_Bronze           86
Iron_Age_Early         40
Neolithic              24
Name: count, dtype: int64


In [137]:
# Ver ejemplos de Bronce que antes estaban mal clasificados
bronce_samples = trt_clean[trt_clean['Cultural_Horizon'].str.contains('Bronze', na=False, case=False)]
print("\n--- Ejemplos de muestras de Bronce y su nueva clasificación ---")
print(bronce_samples[['Cultural_Horizon', 'Absolute_Chronology', 'Chronological_Period_corr']].head(20))


--- Ejemplos de muestras de Bronce y su nueva clasificación ---
     Cultural_Horizon Absolute_Chronology Chronological_Period_corr
15   Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
16   Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
277  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
278  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
279  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
280  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
281  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
282  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
283  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
284  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
285  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
286  Early Bronze Age     2000 - 1500 BCE        Middle_Late_Bronze
287  Early Bronze Age     2000 - 1500 BCE        Mi

In [138]:
def unificar_nombre_periodo(periodo):
    """
    Unifica los nombres de periodo con formato Escala_Período
    Ejemplos: Late_Iron_Age, Early_Bronze_Age, Middle_Late_Bronze_Age
    """
    if periodo == 'Neolithic':
        return 'Neolithic'
    elif periodo == 'Neolithic_Late':
        return 'Late_Neolithic'
    elif periodo == 'Chalcolithic':
        return 'Chalcolithic'
    elif periodo == 'Early_Bronze':
        return 'Early_Bronze_Age'
    elif periodo == 'Middle_Late_Bronze':
        return 'Middle_Late_Bronze_Age'
    elif periodo == 'Iron_Age_Early':
        return 'Early_Iron_Age'
    elif periodo == 'Iron_Age_Late':
        return 'Late_Iron_Age'
    elif periodo == 'Roman':
        return 'Roman_Period'
    else:
        return 'Other'

# Aplicar unificación
trt_clean['Chronological_Period_clean'] = trt_clean['Chronological_Period_corr'].apply(unificar_nombre_periodo)

print("\n--- Periodos UNIFICADOS (formato Escala_Período) ---")
print(trt_clean['Chronological_Period_clean'].value_counts())

# Verificar orden cronológico
orden_cronologico = ['Neolithic', 'Late_Neolithic', 'Chalcolithic', 'Early_Bronze_Age', 
                     'Middle_Late_Bronze_Age', 'Early_Iron_Age', 'Late_Iron_Age', 
                     'Roman_Period', 'Other']

print("\n--- Verificación de orden cronológico ---")
for periodo in orden_cronologico:
    if periodo in trt_clean['Chronological_Period_clean'].values:
        count = (trt_clean['Chronological_Period_clean'] == periodo).sum()
        print(f"  {periodo:25} : {count:4} muestras")


--- Periodos UNIFICADOS (formato Escala_Período) ---
Chronological_Period_clean
Late_Iron_Age             408
Middle_Late_Bronze_Age    357
Late_Neolithic            212
Chalcolithic              127
Other                      92
Early_Bronze_Age           86
Early_Iron_Age             40
Neolithic                  24
Name: count, dtype: int64

--- Verificación de orden cronológico ---
  Neolithic                 :   24 muestras
  Late_Neolithic            :  212 muestras
  Chalcolithic              :  127 muestras
  Early_Bronze_Age          :   86 muestras
  Middle_Late_Bronze_Age    :  357 muestras
  Early_Iron_Age            :   40 muestras
  Late_Iron_Age             :  408 muestras
  Other                     :   92 muestras


In [139]:
# Eliminar columnas que ya no necesitamos
columnas_a_eliminar = [
    'Fecha_Min_BP', 'Fecha_Max_BP',           # versiones antiguas (incorrectas)
    'Periodo_Estandar',                        # versión antigua
    'Chronological_Period',                    # versión antigua (incorrecta)
    'Cultural_Simplified',                     # ya no la necesitamos
    'Med_Basin',                               # duplicado de Mediterranean_Basin
    'Historical_Subperiod' if 'Historical_Subperiod' in trt_clean.columns else '',  # si existe
]

# Eliminar solo las que existen
for col in columnas_a_eliminar:
    if col in trt_clean.columns and col != '':
        trt_clean = trt_clean.drop(columns=[col])

print(f"\n✅ Columnas finales: {len(trt_clean.columns)}")


✅ Columnas finales: 18


In [140]:
print("Columnas disponibles en trt_clean:")
print([col for col in trt_clean.columns if 'Cereal' not in col and 'cereal' not in col])
print("\n¿Hay alguna columna que contenga 'cereal' o 'trigo'?")
cereal_cols = [col for col in trt_clean.columns if 'cereal' in col.lower() or 'trigo' in col.lower() or 'wheat' in col.lower() or 'barley' in col.lower()]
print(cereal_cols)

Columnas disponibles en trt_clean:
['Entry_ID', 'Archaeological_Site', 'IRMS_d13C_Collagen', 'd15N_Collagen', 'Latitude_N', 'Longitude_E', 'General_Category_or_Family', 'Sampled_Element', 'Cultural_Horizon', 'Species', 'Genus', 'Absolute_Chronology', 'Modern_Country', 'Mediterranean_Basin', 'Fecha_Min_BP_corr', 'Fecha_Max_BP_corr', 'Chronological_Period_corr', 'Chronological_Period_clean']

¿Hay alguna columna que contenga 'cereal' o 'trigo'?
[]


In [141]:
def tipo_cereal(genero):
    if pd.isna(genero):
        return 'unknown'
    genero = str(genero).lower()
    if 'triticum' in genero:
        return 'wheat'
    if 'hordeum' in genero:
        return 'barley'
    return 'other'

trt_clean['Cereal'] = trt_clean['Genus'].apply(tipo_cereal)

print("\n--- Distribución de Cereal (recreada) ---")
print(trt_clean['Cereal'].value_counts())


--- Distribución de Cereal (recreada) ---
Cereal
barley    587
wheat     562
other     197
Name: count, dtype: int64


In [142]:
print("\n--- Columna Mediterranean_Basin ---")
if 'Mediterranean_Basin' in trt_clean.columns:
    print(trt_clean['Mediterranean_Basin'].value_counts())
else:
    print("❌ 'Mediterranean_Basin' no existe. Vamos a crearla.")
    
    # Recrear Mediterranean_Basin si no existe
    def cuenca_mediterranea(lat, lon):
        if pd.isna(lat) or pd.isna(lon):
            return 'unknown'
        if not (25 <= lat <= 46 and -10 <= lon <= 42):
            return 'other'
        if lon <= 11:
            return 'Western_Med'
        elif lon <= 23:
            return 'Central_Med'
        else:
            return 'Eastern_Med'
    
    trt_clean['Mediterranean_Basin'] = trt_clean.apply(
        lambda row: cuenca_mediterranea(row['Latitude_N'], row['Longitude_E']), 
        axis=1
    )
    print("\n--- Mediterranean_Basin recreada ---")
    print(trt_clean['Mediterranean_Basin'].value_counts())


--- Columna Mediterranean_Basin ---
Mediterranean_Basin
Central_Med    665
Eastern_Med    461
Western_Med    215
unknown          5
Name: count, dtype: int64


In [143]:
print("\n" + "="*60)
print("DATASET FINAL LISTO PARA MODELIZAR (CORREGIDO)")
print("="*60)

print(f"\n📊 Dimensiones: {trt_clean.shape[0]} muestras, {trt_clean.shape[1]} columnas")

print(f"\n📋 Variables disponibles (solo las útiles para ML):")
for col in ['IRMS_d13C_Collagen', 'd15N_Collagen', 'Latitude_N', 'Longitude_E',
            'Chronological_Period_clean', 'Cereal', 'Mediterranean_Basin', 
            'Genus', 'Species', 'Modern_Country']:
    if col in trt_clean.columns:
        dtype = trt_clean[col].dtype
        nulos = trt_clean[col].isnull().sum()
        unique = trt_clean[col].nunique() if dtype == 'object' else '-'
        print(f"  • {col:30} ({dtype}) - {nulos:3} nulos - {unique} valores únicos")

print(f"\n🎯 VARIABLES TARGET disponibles:")
print(f"  • Clasificación (multiclase): 'Chronological_Period_clean' ({trt_clean['Chronological_Period_clean'].nunique()} periodos)")
print(f"  • Clasificación (binaria):   'Cereal' ({trt_clean['Cereal'].nunique()} tipos)")
print(f"  • Clasificación (multiclase): 'Mediterranean_Basin' ({trt_clean['Mediterranean_Basin'].nunique()} cuencas)")
print(f"  • Regresión:                  'IRMS_d13C_Collagen' (δ13C)")
print(f"  • Regresión:                  'd15N_Collagen' (δ15N)")


DATASET FINAL LISTO PARA MODELIZAR (CORREGIDO)

📊 Dimensiones: 1346 muestras, 19 columnas

📋 Variables disponibles (solo las útiles para ML):
  • IRMS_d13C_Collagen             (float64) -   0 nulos - - valores únicos
  • d15N_Collagen                  (float64) -   0 nulos - - valores únicos
  • Latitude_N                     (float64) -   5 nulos - - valores únicos
  • Longitude_E                    (float64) -   5 nulos - - valores únicos
  • Chronological_Period_clean     (object) -   0 nulos - 8 valores únicos
  • Cereal                         (object) -   0 nulos - 3 valores únicos
  • Mediterranean_Basin            (object) -   0 nulos - 4 valores únicos
  • Genus                          (object) -   0 nulos - 15 valores únicos
  • Species                        (object) -   5 nulos - 24 valores únicos
  • Modern_Country                 (object) -   0 nulos - 8 valores únicos

🎯 VARIABLES TARGET disponibles:
  • Clasificación (multiclase): 'Chronological_Period_clean' (8 peri

In [159]:
# Guardar versión para modelos (sin cambios)
trt_clean.to_csv('Neo_Met_plants.csv', index=False)

# Guardar versión CON sitio arqueológico
trt_clean.to_csv('Med_Plants.csv', index=False)

print("\n✅ Datasets guardados:")
print("   - Neo_Met_plants.csv (para modelos, sin cambios)")
print("   - Med_Plants.csv (con columna 'Archaeological_Site')")
print(f"\n📊 Dimensiones finales: {trt_clean.shape}")
print(f"📋 Columnas: {trt_clean.columns.tolist()}")


✅ Datasets guardados:
   - Neo_Met_plants.csv (para modelos, sin cambios)
   - Med_Plants.csv (con columna 'Archaeological_Site')

📊 Dimensiones finales: (1346, 19)
📋 Columnas: ['Entry_ID', 'Archaeological_Site', 'IRMS_d13C_Collagen', 'd15N_Collagen', 'Latitude_N', 'Longitude_E', 'General_Category_or_Family', 'Sampled_Element', 'Cultural_Horizon', 'Species', 'Genus', 'Absolute_Chronology', 'Modern_Country', 'Mediterranean_Basin', 'Fecha_Min_BP_corr', 'Fecha_Max_BP_corr', 'Chronological_Period_corr', 'Chronological_Period_clean', 'Cereal']


In [160]:
df = pd.read_csv('Neo_Met_plants.csv')
df2 = pd.read_csv('Med_Plants.csv')


In [161]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1346 entries, 0 to 1345
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Entry_ID                    1346 non-null   int64  
 1   Archaeological_Site         1334 non-null   object 
 2   IRMS_d13C_Collagen          1346 non-null   float64
 3   d15N_Collagen               1346 non-null   float64
 4   Latitude_N                  1341 non-null   float64
 5   Longitude_E                 1341 non-null   float64
 6   General_Category_or_Family  1346 non-null   object 
 7   Sampled_Element             1267 non-null   object 
 8   Cultural_Horizon            1064 non-null   object 
 9   Species                     1341 non-null   object 
 10  Genus                       1346 non-null   object 
 11  Absolute_Chronology         1254 non-null   object 
 12  Modern_Country              1346 non-null   object 
 13  Mediterranean_Basin         1346 

In [162]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1346 entries, 0 to 1345
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Entry_ID                    1346 non-null   int64  
 1   Archaeological_Site         1334 non-null   object 
 2   IRMS_d13C_Collagen          1346 non-null   float64
 3   d15N_Collagen               1346 non-null   float64
 4   Latitude_N                  1341 non-null   float64
 5   Longitude_E                 1341 non-null   float64
 6   General_Category_or_Family  1346 non-null   object 
 7   Sampled_Element             1267 non-null   object 
 8   Cultural_Horizon            1064 non-null   object 
 9   Species                     1341 non-null   object 
 10  Genus                       1346 non-null   object 
 11  Absolute_Chronology         1254 non-null   object 
 12  Modern_Country              1346 non-null   object 
 13  Mediterranean_Basin         1346 

In [148]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset final
df = pd.read_csv('NeoM-plants.csv')

# Configuración de gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Dataset cargado")
print(f"Dimensiones: {df.shape}")
print(f"Periodos: {df['Chronological_Period_clean'].nunique()}")
print(f"Cereales: {df['Cereal'].unique()}")
print(f"Cuencas: {df['Mediterranean_Basin'].unique()}")

✅ Dataset cargado
Dimensiones: (1346, 18)
Periodos: 8
Cereales: ['other' 'barley' 'wheat']
Cuencas: ['Western_Med' 'unknown' 'Central_Med' 'Eastern_Med']


In [149]:
# Filtrar muestras donde Cereal = 'other'
cereal_other = df[df['Cereal'] == 'other']

print("="*60)
print("MUESTRAS CON Cereal = 'other'")
print("="*60)
print(f"Total: {len(cereal_other)} muestras ({len(cereal_other)/len(df)*100:.1f}%)")

print("\n--- Géneros (Genus) en 'other' ---")
print(cereal_other['Genus'].value_counts())

print("\n--- Familias (General_Category_or_Family) en 'other' ---")
print(cereal_other['General_Category_or_Family'].value_counts())

print("\n--- Especies (Species) en 'other' ---")
print(cereal_other['Species'].value_counts().head(20))

MUESTRAS CON Cereal = 'other'
Total: 197 muestras (14.6%)

--- Géneros (Genus) en 'other' ---
Genus
Lens          60
Vicia         40
\tLathyrus    30
Pisum         21
Vicia         20
Panicum       16
Avena          5
Avena          2
Olea           1
Linum          1
Pistacia       1
Name: count, dtype: int64

--- Familias (General_Category_or_Family) en 'other' ---
General_Category_or_Family
Fabaceae         171
Poaceae           23
Oleaceae           1
Linaceae           1
Anacardiaceae      1
Name: count, dtype: int64

--- Especies (Species) en 'other' ---
Species
culinaris        60
faba             42
sativum          20
clymenum         19
milliaceum       15
ervilia          14
sativus          11
sativa            9
sp.               3
europaea          1
usitatissimum     1
miliaceum         1
vera              1
Name: count, dtype: int64


In [150]:
# Filtrar muestras donde Mediterranean_Basin = 'unknown'
basin_unknown = df[df['Mediterranean_Basin'] == 'unknown']

print("\n" + "="*60)
print("MUESTRAS CON Mediterranean_Basin = 'unknown'")
print("="*60)
print(f"Total: {len(basin_unknown)} muestras ({len(basin_unknown)/len(df)*100:.1f}%)")

print("\n--- Coordenadas de estas muestras ---")
print(basin_unknown[['Latitude_N', 'Longitude_E', 'Modern_Country']])

print("\n--- Países de estas muestras ---")
print(basin_unknown['Modern_Country'].value_counts())


MUESTRAS CON Mediterranean_Basin = 'unknown'
Total: 5 muestras (0.4%)

--- Coordenadas de estas muestras ---
    Latitude_N  Longitude_E Modern_Country
9          NaN          NaN          Italy
10         NaN          NaN          Italy
11         NaN          NaN          Italy
12         NaN          NaN          Italy
13         NaN          NaN          Italy

--- Países de estas muestras ---
Modern_Country
Italy    5
Name: count, dtype: int64


In [151]:
# ¿Muestras que son both 'other' y 'unknown'?
both = df[(df['Cereal'] == 'other') & (df['Mediterranean_Basin'] == 'unknown')]
print("\n" + "="*60)
print("MUESTRAS CON Cereal='other' Y Mediterranean_Basin='unknown'")
print("="*60)
print(f"Total: {len(both)} muestras")
if len(both) > 0:
    print(both[['Genus', 'Modern_Country', 'Latitude_N', 'Longitude_E']])


MUESTRAS CON Cereal='other' Y Mediterranean_Basin='unknown'
Total: 3 muestras
     Genus Modern_Country  Latitude_N  Longitude_E
11  Vicia           Italy         NaN          NaN
12   Olea           Italy         NaN          NaN
13  Avena           Italy         NaN          NaN


In [152]:
# Eliminar filas sin cuenca definida
df_clean = df[df['Mediterranean_Basin'] != 'unknown'].copy()
print(f"✅ Eliminadas 5 muestras sin cuenca. Nuevo tamaño: {len(df_clean)}")

✅ Eliminadas 5 muestras sin cuenca. Nuevo tamaño: 1341


In [153]:
# Esta función examina el género (Genus) y la especie (Species) de cada planta
# y decide si es un cereal (trigo, cebada, mijo, avena) o no.
# 
# ¿Por qué? Los cereales tienen diferentes necesidades hídricas y de fertilización
# que las legumbres o los árboles frutales. Separarlos nos ayudará en los modelos.

def clasificar_cereal(genus, species):
    # Convertimos a minúsculas para evitar problemas con mayúsculas/minúsculas
    genus = str(genus).lower() if pd.notna(genus) else ''
    species = str(species).lower() if pd.notna(species) else ''
    
    # Hordeum = cebada (es el género botánico)
    if 'hordeum' in genus:
        return 'barley'
    
    # Triticum = trigo (incluye durum, aestivum, monococcum, dicoccum)
    if 'triticum' in genus:
        return 'wheat'
    
    # Panicum = mijo (cultivo de verano, tolerante a sequía)
    if 'panicum' in genus:
        return 'broomcorn_millet'
    
    # Avena = avena (menos común en contextos antiguos)
    if 'avena' in genus:
        return 'oat'
    
    # Si no es ninguno de los anteriores, no es un cereal
    return 'not_cereal'

# Aplicamos la función a cada fila del dataframe
# axis=1 significa "aplicar a cada fila" (en lugar de a cada columna)
df_clean['Cereal_Type'] = df_clean.apply(
    lambda row: clasificar_cereal(row['Genus'], row['Species']), 
    axis=1
)

# Mostramos cuántas muestras tiene cada tipo de cereal
print("\n--- CLASIFICACIÓN DE CEREALES ---")
print(df_clean['Cereal_Type'].value_counts())


--- CLASIFICACIÓN DE CEREALES ---
Cereal_Type
barley              591
wheat               556
not_cereal          172
broomcorn_millet     16
oat                   6
Name: count, dtype: int64


In [154]:
# Esta función clasifica las plantas que NO son cereales
# Incluye legumbres (lentejas, habas, guisantes...) y otros cultivos (olivo, lino, pistacho)
#
# ¿Por qué? Las legumbres fijan nitrógeno atmosférico (afecta al δ15N),
# mientras que el olivo o la vid tienen metabolismos diferentes.

def clasificar_pulse(genus, species):
    genus = str(genus).lower() if pd.notna(genus) else ''
    species = str(species).lower() if pd.notna(species) else ''
    
    # --- LEGUMBRES (fabaceae) ---
    # Lens = lenteja
    if 'lens' in genus:
        return 'lentil'
    
    # Vicia + 'faba' = haba (Vicia faba)
    if 'vicia' in genus and 'faba' in species:
        return 'faba_bean'
    
    # Otras Vicia (no faba) = veza amarga (Vicia ervilia u otras)
    if 'vicia' in genus:
        return 'vetch'
    
    # Pisum = guisante
    if 'pisum' in genus:
        return 'pea'
    
    # Lathyrus = almorta o guija (Lathyrus sativus/clymenum)
    if 'lathyrus' in genus:
        return 'grass_pea'
    
    # --- OTROS CULTIVOS ---
    # Olea = olivo (producción de aceite)
    if 'olea' in genus:
        return 'olive'
    
    # Linum = lino (fibra y semillas oleaginosas)
    if 'linum' in genus:
        return 'flax'
    
    # Pistacia = pistacho o terebinto
    if 'pistacia' in genus:
        return 'pistachio'
    
    # Si no es ninguna de las anteriores, lo etiquetamos como 'other_pulse'
    return 'other_pulse'

# Aplicamos la función
df_clean['Pulse_Type'] = df_clean.apply(
    lambda row: clasificar_pulse(row['Genus'], row['Species']), 
    axis=1
)

# Ahora tenemos un problema: las muestras que son cereal tienen un valor en Pulse_Type
# que no es correcto (por ejemplo, una cebada no es una legumbre).
# Vamos a corregirlo: si es cereal, Pulse_Type debe ser 'not_pulse'

df_clean.loc[df_clean['Cereal_Type'] != 'not_cereal', 'Pulse_Type'] = 'not_pulse'

# Y viceversa: si es legumbre/otro, Cereal_Type debe ser 'not_cereal'
# (esto ya lo está porque la función clasificar_cereal devuelve 'not_cereal' para estos casos)

print("\n--- CLASIFICACIÓN DE LEGUMBRES Y OTROS ---")
print(df_clean['Pulse_Type'].value_counts())


--- CLASIFICACIÓN DE LEGUMBRES Y OTROS ---
Pulse_Type
not_pulse    1169
lentil         60
faba_bean      41
grass_pea      30
pea            21
vetch          18
flax            1
pistachio       1
Name: count, dtype: int64


In [155]:
# Comprobamos que ninguna muestra sea simultáneamente cereal Y legumbre
conflictos = df_clean[(df_clean['Cereal_Type'] != 'not_cereal') & (df_clean['Pulse_Type'] != 'not_pulse')]
print(f"\n--- SOLAPAMIENTOS (no debería haber ninguno) ---")
print(f"Muestras con doble clasificación: {len(conflictos)}")

if len(conflictos) > 0:
    print(conflictos[['Genus', 'Species', 'Cereal_Type', 'Pulse_Type']])


--- SOLAPAMIENTOS (no debería haber ninguno) ---
Muestras con doble clasificación: 0


In [156]:
# Guardamos la versión definitiva para los modelos
df_clean.to_csv('Neo_Met_plants.csv', index=False)

print("\n" + "="*60)
print("RESUMEN FINAL DEL DATASET")
print("="*60)
print(f"Dimensiones: {df_clean.shape[0]} muestras, {df_clean.shape[1]} columnas")
print(f"\nVariables de clasificación de plantas:")
print(f"  • Cereal_Type : {df_clean['Cereal_Type'].nunique()} tipos")
print(f"  • Pulse_Type  : {df_clean['Pulse_Type'].nunique()} tipos")


RESUMEN FINAL DEL DATASET
Dimensiones: 1341 muestras, 20 columnas

Variables de clasificación de plantas:
  • Cereal_Type : 5 tipos
  • Pulse_Type  : 8 tipos


In [157]:
df = pd.read_csv('Neo_Met_plants.csv')

In [158]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341 entries, 0 to 1340
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Entry_ID                    1341 non-null   int64  
 1   IRMS_d13C_Collagen          1341 non-null   float64
 2   d15N_Collagen               1341 non-null   float64
 3   Latitude_N                  1341 non-null   float64
 4   Longitude_E                 1341 non-null   float64
 5   General_Category_or_Family  1341 non-null   object 
 6   Sampled_Element             1267 non-null   object 
 7   Cultural_Horizon            1064 non-null   object 
 8   Species                     1336 non-null   object 
 9   Genus                       1341 non-null   object 
 10  Absolute_Chronology         1254 non-null   object 
 11  Modern_Country              1341 non-null   object 
 12  Mediterranean_Basin         1341 non-null   object 
 13  Fecha_Min_BP_corr           1254 